# CE-only: is it the objective, or the schedule around it?

`--lambdas 0 0 0 1` keeps everything BaCP has except the contrastive terms.
Same SGD 0.1, same two-view SimCLR augmentation, same 50 + 25 epochs, same
student loaded from the dense checkpoint, same seeds.

| comparison | isolates |
|---|---|
| BaCP − CE-only | the contrastive objective, alone |
| CE-only − I.P. | learning rate + augmentation + recovery dynamics |

Nine runs. Two cells in the damaged regime where the grid effect lives
(ResNet-34 and ResNet-50, magnitude, 0.999) and one intact-regime control
(ResNet-34, magnitude, 0.95) where the grid shows nothing.

Magnitude is deliberate: both arms then score the identical `|w|` criterion, so
no arm-specific information enters the mask and the learning-rate objection has
no purchase on the comparison.

Records carry the `.ceonly` suffix and sit beside the grid rather than in it.


In [ ]:
import sys, pathlib

here = pathlib.Path.cwd()
while not (here / '.git').exists() and here != here.parent:
    here = here.parent
sys.path.insert(0, str(here / 'project' / 'test_notebooks'))

import nb_common as nb
info = nb.setup()


## Plan

In [ ]:
PRUNER = 'magnitude'
SEEDS  = (1, 2, 3)
GPU    = 0

# (model, sparsity) -- 0.999 is the damaged regime, 0.95 the intact control
CELLS = [('resnet34', 0.999), ('resnet50', 0.999), ('resnet34', 0.95)]

plan = []
for model, sp in CELLS:
    for s in SEEDS:
        plan.append(nb.make_cell(model, 'bacp', seed=s, pruner=PRUNER, sparsity=sp,
                                 variant='ceonly', lambdas=[0.0, 0.0, 0.0, 1.0]))

# the lambdas must be the ONLY departure from the reported BaCP arm
for c in plan:
    ref = nb.FAMILIES[c['model_name']]['bacp']
    for k in ('learning_rate', 'epochs', 'epochs_ft', 'delta_T', 'sparsity_scheduler',
              'recovery_epochs', 'val_split', 'prune_task_head', 'wanda_group',
              'optimizer_type', 'optimizer_type_ft', 'learning_rate_ft',
              'batch_size', 'num_classes', 'dataset_name', 'tau',
              'contrastive_mode', 'proj_mode', 'n_views'):
        if k in ref:
            assert c['config'][k] == ref[k], (c['key'], k, c['config'][k], ref[k])
    assert c['config']['lambdas'] == [0.0, 0.0, 0.0, 1.0], c['config']['lambdas']
    assert c['config']['epochs'] == 50 and c['config']['epochs_ft'] == 25
    assert c['key'].endswith('.ceonly'), c['key']

assert len(plan) == 9
print('%d runs' % len(plan))
for c in plan:
    print('   ', c['key'])
print('est ~%.0f min' % sum(8.1 if c['model_name'] == 'resnet34' else 12.6 for c in plan))
assert nb.sanity_check(plan), 'sanity check failed'


## Run

In [ ]:
nb.run_group(plan, gpu=GPU)


## Verdict

In [ ]:
import json, glob, os, statistics as st

root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    try:
        r = json.load(open(f, encoding='utf-8'))
    except Exception:
        continue
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok':
        acc.setdefault(k, []).append(r.get('test_acc_exact_pct') or r.get('test_acc_pct'))

def m(key):
    v = [x for x in acc.get(key, []) if x is not None]
    return (st.mean(v), len(v)) if v else None

print('%-26s %9s %9s %9s   %9s %9s' %
      ('cell', 'I.P.', 'CE-only', 'BaCP', 'BaCP-CE', 'CE-I.P.'))
print('-' * 80)
for model, sp in CELLS:
    base = '%s.cifar10.s%s.%s' % (model, sp, PRUNER)
    got = {}
    for lbl, key in (('ip', 'static.prune.%s.seed%%d.ft25' % base),
                     ('ce', 'static.bacp.%s.seed%%d.ceonly' % base),
                     ('bacp', 'static.bacp.%s.seed%%d' % base)):
        vals = [acc.get(key % s, [None])[0] for s in SEEDS]
        vals = [v for v in vals if v is not None]
        got[lbl] = (st.mean(vals), len(vals)) if vals else None
    f = lambda v: '    --   ' if v is None else '%6.2f%s  ' % (v[0], '*' if v[1] < 3 else ' ')
    d1 = '%+9.2f' % (got['bacp'][0] - got['ce'][0]) if got['bacp'] and got['ce'] else '    --   '
    d2 = '%+9.2f' % (got['ce'][0] - got['ip'][0]) if got['ce'] and got['ip'] else '    --   '
    print('%-26s %9s %9s %9s   %9s %9s'
          % ('%s %s' % (model, sp), f(got['ip']), f(got['ce']), f(got['bacp']), d1, d2))
print()
print('BaCP-CE is the contrastive objective on its own.')
print('CE-I.P. is what the learning rate and augmentation were already buying.')
print('* = fewer than 3 seeds')
